# Craft My Book

Uses the `llm` package in `src/` (OpenAI provider) to draft book content.

In [1]:
import sys
from pathlib import Path


sys.path.insert(0, str(Path.cwd() / "src"))

from llm import get_client, chat
import config

In [2]:
client = get_client(config.LLM_PROVIDER)
model = config.LLM_MODEL

In [3]:
prompt = "Write an engaging opening paragraph for a book about ..."

response = chat(client, model, prompt)
print(response)

Sure! Could you please provide me with the specific theme or topic you'd like the book to be about?


## Module 1.2 — Speech Processing (Whisper)

Turns a lecture recording — video or audio — into a structured, timestamped,
domain-aware transcript (`src/ingestion/speech.py`, `src/ingestion/vocab.py`).
Requires the `ffmpeg` binary on PATH and `faster-whisper` installed
(`pip install -r requirements.txt`).

All the tunable choices — Whisper model size, LLM provider/model, output dir — live in
`src/config/settings.py`, not scattered across cells/function signatures. Whisper
currently defaults to `"small"` (fast, fully local-cacheable) for local iteration;
bump `WHISPER_MODEL_SIZE` to `"large-v3"` in that file for real lecture-quality
transcription once you have a stable connection (and ideally a GPU) — see
`ingestion/speech.py` for why domain accuracy matters there.

Flow: `extract_vocab` (LLM pass over slide titles/filenames/headings — vocab is
per-corpus, never hardcoded) → `extract_audio` (ffmpeg; handles video or audio
sources) → `transcribe_audio` (faster-whisper, vocab-primed, VAD-filtered, word
timestamps, no cross-segment conditioning) → `clean_transcript` (conservative LLM
pass — fixes terms/punctuation, changes nothing else) → JSON.

In [4]:
from ingestion import extract_audio, transcribe_audio, clean_transcript, process_source, extract_vocab, vocab_material_from_filenames, bootstrap_vocab_from_audio

# data/harvard-speech.wav: ~34s of real spoken English (public-domain Harvard sentences
# test corpus) - use this to smoke-test the pipeline locally before pointing it at a
# real lecture.
source_path = "data/harvard-speech.wav"

# Preferred: derive vocab from material that already exists around the recording
# (slide titles / filenames here; could also be PDF headings).
sibling_files: list[str] = []  # no slides for this sample -> falls through to bootstrap
vocab_material = vocab_material_from_filenames(sibling_files)
vocab = extract_vocab(vocab_material, client, model)

# Fallback: audio is the ONLY source (no slides/useful filenames) - bootstrap vocab from
# a fast draft transcription pass instead of skipping priming altogether.
if not vocab:
    audio_path = extract_audio(source_path)
    vocab = bootstrap_vocab_from_audio(audio_path, client, model)

vocab

/opt/homebrew/Caskroom/miniforge/base/envs/rl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['birch canoe',
 'depth',
 'chicken leg',
 'rice',
 'round bowls',
 'juice',
 'lemons',
 'punch',
 'parked truck',
 'hogs',
 'chopped corn',
 'garbage',
 'steady work',
 'large size',
 'stockings']

In [5]:
# Step by step (useful while inspecting intermediate output)
audio_path = extract_audio(source_path)  # works for video or audio sources
raw_transcript = transcribe_audio(audio_path, vocab=vocab)
cleaned_transcript = clean_transcript(raw_transcript, client, model)

print(f"{len(cleaned_transcript.segments)} segments, {cleaned_transcript.duration:.0f}s")
cleaned_transcript.segments[0]

4 segments, 34s


Segment(start=np.float64(0.11), end=np.float64(11.33), text='birch canoe, smooth planks, dark blue background, easy to tell the depth of a well these days.', words=[Word(start=np.float64(0.11), end=np.float64(1.01), text='birch'), Word(start=np.float64(1.01), end=np.float64(1.35), text='canoe,'), Word(start=np.float64(1.61), end=np.float64(1.93), text='smooth'), Word(start=np.float64(1.93), end=np.float64(3.11), text='planks,'), Word(start=np.float64(3.37), end=np.float64(5.17), text='dark'), Word(start=np.float64(5.17), end=np.float64(5.73), text='blue'), Word(start=np.float64(5.73), end=np.float64(6.27), text='background,'), Word(start=np.float64(6.81), end=np.float64(8.19), text='easy'), Word(start=np.float64(8.19), end=np.float64(8.43), text='to'), Word(start=np.float64(8.43), end=np.float64(8.65), text='tell'), Word(start=np.float64(8.65), end=np.float64(8.79), text='the'), Word(start=np.float64(8.79), end=np.float64(9.07), text='depth'), Word(start=np.float64(9.07), end=np.float6

In [6]:
# Or, end to end in one call: source -> audio -> transcribe -> clean -> saved JSON
transcript = process_source(source_path, vocab=vocab, client=client, clean_model=model)
transcript.to_json  # already written to output/transcripts/lecture5.json by process_source

<bound method Transcript.to_json of Transcript(source='harvard-speech.16k.wav', language='en', duration=33.623125, segments=[Segment(start=np.float64(0.11), end=np.float64(11.33), text='Birch canoe, smooth planks, dark blue background, easy to tell the depth of a well these days.', words=[Word(start=np.float64(0.11), end=np.float64(1.01), text='birch'), Word(start=np.float64(1.01), end=np.float64(1.35), text='canoe,'), Word(start=np.float64(1.61), end=np.float64(1.93), text='smooth'), Word(start=np.float64(1.93), end=np.float64(3.11), text='planks,'), Word(start=np.float64(3.37), end=np.float64(5.17), text='dark'), Word(start=np.float64(5.17), end=np.float64(5.73), text='blue'), Word(start=np.float64(5.73), end=np.float64(6.27), text='background,'), Word(start=np.float64(6.81), end=np.float64(8.19), text='easy'), Word(start=np.float64(8.19), end=np.float64(8.43), text='to'), Word(start=np.float64(8.43), end=np.float64(8.65), text='tell'), Word(start=np.float64(8.65), end=np.float64(8.7